# Practical 1 — Superstore product mix

R Jupyter version of the course material. Select an **R** kernel in Positron and run cells from top to bottom, starting with the setup cell. Keep the complete course folder, data and images together.

Generated from the matching QMD; make teaching edits in the QMD and regenerate. The published HTML/PDF retains the full course styling.


In [ ]:
# Notebook sessions may start in the notebook folder.
# Find the opened course folder so the original data/... paths work unchanged.
.course_root <- normalizePath(getwd(), winslash = "/")
while (!file.exists(file.path(.course_root, "_quarto.yml")) ||
       !dir.exists(file.path(.course_root, "data"))) {
  .course_parent <- dirname(.course_root)
  if (identical(.course_parent, .course_root)) {
    stop("Open the complete course folder containing _quarto.yml and data/.")
  }
  .course_root <- .course_parent
}
setwd(.course_root)
rm(.course_root)
if (exists(".course_parent", inherits = FALSE)) rm(.course_parent)


In [ ]:
#| label: categorical-import
source("lessons/_includes/course-tools.R")
lines <- read.csv("data/business/superstore-lines.csv",
                  stringsAsFactors = FALSE, colClasses = c(postal_code = "character"))
orders <- make_orders(lines)
c(lines = nrow(lines), orders = nrow(orders), customers = n_distinct(lines$customer_id))
colSums(is.na(lines))


- `make_orders()` verifies consistent attributes and combines lines with the same order ID.
- It does not count a multi-line order several times.
- IDs and postal codes remain labels.


In [ ]:
knitr::kable(head(lines[c("order_id", "category", "sub_category", "segment", "region")]))


## 3. Describe the category mix


In [ ]:
#| fig-alt: "Product-line counts by category, led by Office Supplies."
category_mix <- lines %>% count(category, name = "line_count") %>%
  mutate(line_pct = 100 * line_count / sum(line_count))
knitr::kable(category_mix, digits = 1)

ggplot(category_mix, aes(reorder(category, line_count), line_count)) +
  geom_col(fill = petra_ink) + coord_flip() +
  labs(x = NULL, y = "Product lines")


- `count()` produces frequencies.
- The denominator is all product lines in the extract.
- Office Supplies dominates line count; this does not establish the largest revenue or profit contribution.
- Categories have no required numerical order, so a cumulative category percentage would depend on an arbitrary arrangement.

### Change the business measure


In [ ]:
category_sales <- lines %>% group_by(category) %>%
  summarise(total_sales = sum(sales), total_units = sum(quantity), .groups = "drop") %>%
  mutate(sales_pct = 100 * total_sales / sum(total_sales))
knitr::kable(category_sales, digits = 1)


- A line can contain several units and very different sales values.
- Line share, unit share and revenue share answer different questions.
- Technology leads revenue here even though it has fewer lines than Office Supplies.

**Pause:** Which measure would you use to discuss warehouse picking activity? Which would you use to discuss revenue? Explain the limits of each.

## 4. Compare customer segments


In [ ]:
counts <- xtabs(~ segment + category, data = lines)
row_percent <- 100 * prop.table(counts, margin = 1)
knitr::kable(counts)
knitr::kable(row_percent, digits = 1)


- Rows: customer segments.
- Columns: product categories.
- Each percentage row totals 100%.
- A cell answers: “Within this segment’s product lines, what percentage belong to this category?”
- Dividing by a column total answers a different question: segment mix within a category.


In [ ]:
#| fig-alt: "One hundred percent stacked bars compare category mix within each customer segment."
plot_mix <- lines %>% count(segment, category) %>%
  group_by(segment) %>% mutate(percent = 100 * n / sum(n)) %>% ungroup()
ggplot(plot_mix, aes(segment, percent, fill = category)) +
  geom_col() + scale_fill_manual(values = petra_categories) +
  labs(x = NULL, y = "Product lines within segment (%)", fill = "Category")


- Every segment bar reaches 100%, even though segment counts differ.
- State both percentages and denominators when reporting a comparison.
- A difference of 5 percentage points is not automatically a 5% relative increase.

## 5. Use an ordered cumulative summary

> What share of orders is dispatched within each number of days?


In [ ]:
#| fig-alt: "Cumulative percentage of orders dispatched within each number of days."
shipping <- orders %>% count(ship_days, name = "order_count") %>%
  arrange(ship_days) %>%
  mutate(percent = 100 * order_count / sum(order_count), cumulative_pct = cumsum(percent))
knitr::kable(shipping, digits = 1)
ggplot(shipping, aes(ship_days, cumulative_pct)) +
  geom_step(colour = petra_ink) + geom_point(colour = petra_ink) +
  labs(x = "Days from order to dispatch", y = "Orders dispatched within this duration (%)")


- Durations have a natural order, so cumulative percentages are meaningful.
- Use one record per order; counting lines would give large baskets extra weight.
- Dispatch time is not delivery time.
- Observed lag is not the same as an advertised service guarantee.

## 6. Interpret a returns table correctly


In [ ]:
return_mix <- orders %>% count(return_status, name = "order_count") %>%
  mutate(percent = 100 * order_count / sum(order_count))
knitr::kable(return_mix, digits = 2)


- An order is **Listed return** when its ID appears in the workbook’s Returns sheet.
- The unit and denominator are orders, not products, returned units or customers.
- **Not listed** is absence from this source table; it is not independently confirmed non-return.
- Do not use the table to infer refund amounts or to adjust recorded profit.

## 7. Independent business briefing

The regional manager asks for a one-page summary.

1.  Build a region frequency table using **orders**, with percentages.
2.  Compare shipping-mode percentages **within each region**.
3.  Show one chart that makes this comparison readable.
4.  Calculate the percentage dispatched within four days, overall and by region.
5.  Report the listed-return percentage for each segment, using orders.
6.  Write four bullets: finding, numerical evidence, denominator and limitation.


**Manual setup — run once in the R Console if needed.** This cell is excluded from Run All.

```r
#| eval: false
# Use orders for these questions.
# Start with count(region), then xtabs(~ region + ship_mode, data = orders).
# For a group proportion: mean(ship_days <= 4) within each region.
```


**Check your reasoning:**

- Would line-based results give every order equal weight?
- Are you interpreting dispatch rather than delivery?
- Does the reader know whether a percentage is within a row or within the full table?

## 8. Continue

- Keep your tables and briefing in your Positron workspace.
- Continue to [Practical 2: Order size](https://msfasha.github.io/307312-Applied-Statistics/extended-cases/superstore/practical-02-numerical-business-data.html).
- Source definitions and extract limits: `data/business/README.md`.
